# APAC E-commerce Strategy & Operations Analysis
**Author:** Kalaivani Mathivanan  
**Tools:** Python · SQL (SQLite) · pandas · matplotlib · seaborn  
**Dataset:** 15,000 APAC e-commerce orders across 6 markets (Jan–Dec 2024)  
**Objective:** Surface actionable insights on revenue performance, operational efficiency, and GTM priorities across APAC markets to support Strategy & Operations decision-making.

---

### Business Questions This Analysis Answers
1. Which APAC markets are driving the most revenue — and which have the highest growth trajectory?
2. Which categories and channels are underperforming relative to their potential?
3. Where are operational bottlenecks (fulfillment SLA, return rates) hurting customer experience?
4. Where should we focus GTM investment in H1 2025?

---

## 1. Setup & Data Loading

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'font.family': 'DejaVu Sans', 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.facecolor': '#FAFAFA'})

COUNTRY_COLORS = {
    'Singapore': '#1A73E8', 'Indonesia': '#34A853',
    'Philippines': '#EA4335', 'Vietnam': '#FBBC04',
    'Thailand': '#9C27B0', 'Malaysia': '#00ACC1'
}

# Load data into SQLite
conn = sqlite3.connect(':memory:')
df = pd.read_csv('../data/apac_ecommerce_orders.csv', parse_dates=['order_date'])
df.to_sql('orders', conn, index=False, if_exists='replace')
print(f'Loaded {len(df):,} orders | Revenue: ${df["revenue_usd"].sum():,.0f} USD')
df.head(3)

## 2. Data Validation (SQL)

In [ ]:
# SQL: completeness and consistency checks
validation_sql = """
SELECT
    COUNT(*)                                       AS total_orders,
    COUNT(DISTINCT country)                        AS markets,
    MIN(order_date)                                AS date_from,
    MAX(order_date)                                AS date_to,
    SUM(CASE WHEN revenue_usd IS NULL THEN 1 END)  AS null_revenue,
    SUM(CASE WHEN ABS(revenue_usd - unit_price_usd * quantity) > 0.05 THEN 1 END) AS inconsistent_rows
FROM orders
"""
pd.read_sql_query(validation_sql, conn)

## 3. Revenue Performance by Market (SQL)

In [ ]:
market_sql = """
SELECT
    country,
    COUNT(order_id)                                              AS orders,
    ROUND(SUM(revenue_usd), 0)                                   AS total_revenue_usd,
    ROUND(AVG(revenue_usd), 2)                                   AS aov_usd,
    ROUND(SUM(revenue_usd)*100.0 / SUM(SUM(revenue_usd)) OVER (), 1) AS revenue_share_pct,
    ROUND(SUM(returned)*100.0 / COUNT(*), 1)                     AS return_rate_pct,
    ROUND(AVG(delivery_days), 1)                                 AS avg_delivery_days
FROM orders
GROUP BY country
ORDER BY total_revenue_usd DESC
"""
df_market = pd.read_sql_query(market_sql, conn)
df_market

## 4. Category Performance (SQL)

In [ ]:
cat_sql = """
SELECT
    category,
    COUNT(order_id)                                              AS orders,
    ROUND(SUM(revenue_usd), 0)                                   AS total_revenue_usd,
    ROUND(AVG(revenue_usd), 2)                                   AS aov_usd,
    ROUND(SUM(revenue_usd)*100.0 / SUM(SUM(revenue_usd)) OVER (), 1) AS revenue_share_pct,
    ROUND(SUM(returned)*100.0 / COUNT(*), 1)                     AS return_rate_pct
FROM orders
GROUP BY category
ORDER BY total_revenue_usd DESC
"""
df_cat = pd.read_sql_query(cat_sql, conn)
df_cat

## 5. Operational KPIs — Fulfillment & Returns (SQL)

In [ ]:
ops_sql = """
SELECT
    country,
    ROUND(AVG(delivery_days), 1)  AS avg_delivery_days,
    ROUND(SUM(CASE WHEN delivery_days > 4 THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS sla_breach_pct,
    ROUND(SUM(returned)*100.0/COUNT(*), 1) AS return_rate_pct,
    CASE WHEN AVG(delivery_days) > 4 THEN 'ACTION REQUIRED' ELSE 'ON TRACK' END AS sla_status
FROM orders
GROUP BY country
ORDER BY avg_delivery_days DESC
"""
df_ops = pd.read_sql_query(ops_sql, conn)
df_ops

## 6. Channel & Payment Method Analysis (SQL)

In [ ]:
channel_sql = """
SELECT
    channel,
    COUNT(order_id)                              AS orders,
    ROUND(SUM(revenue_usd), 0)                   AS revenue_usd,
    ROUND(AVG(revenue_usd), 2)                   AS aov_usd,
    ROUND(SUM(is_repeat_customer)*100.0/COUNT(*), 1) AS repeat_customer_pct
FROM orders
GROUP BY channel
ORDER BY revenue_usd DESC
"""
df_channel = pd.read_sql_query(channel_sql, conn)
df_channel

## 7. Quarterly Growth by Market (GTM Insight)

In [ ]:
qtr_sql = """
SELECT country, quarter,
       ROUND(SUM(revenue_usd), 0) AS quarterly_gmv,
       COUNT(order_id) AS orders
FROM orders
GROUP BY country, quarter
ORDER BY country, quarter
"""
df_qtr = pd.read_sql_query(qtr_sql, conn)
df_qtr_pivot = df_qtr.pivot(index='country', columns='quarter', values='quarterly_gmv')
df_qtr_pivot['Q4_vs_Q1_growth_pct'] = ((df_qtr_pivot['Q4'] - df_qtr_pivot['Q1']) / df_qtr_pivot['Q1'] * 100).round(1)
df_qtr_pivot.sort_values('Q4_vs_Q1_growth_pct', ascending=False)

## 8. Dashboard — All Key Metrics

In [ ]:
# Monthly trend data
monthly_sql = """
SELECT month, COUNT(order_id) AS orders,
       ROUND(SUM(revenue_usd),0) AS gmv_usd,
       ROUND(AVG(revenue_usd),2) AS aov,
       ROUND(SUM(returned)*100.0/COUNT(*),1) AS return_rate
FROM orders GROUP BY month ORDER BY month
"""
df_monthly = pd.read_sql_query(monthly_sql, conn)

social_sql = """
SELECT country,
       ROUND(SUM(CASE WHEN channel='Social Commerce' THEN revenue_usd ELSE 0 END)*100.0/SUM(revenue_usd),1) AS social_pct
FROM orders GROUP BY country ORDER BY social_pct DESC
"""
df_social = pd.read_sql_query(social_sql, conn)

fig = plt.figure(figsize=(20, 22))
fig.suptitle('APAC E-commerce Strategy & Operations Dashboard — 2024',
             fontsize=20, fontweight='bold', y=0.99, color='#202124')
gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

# Panel 1: Revenue by market
ax1 = fig.add_subplot(gs[0, 0])
colors_bar = [COUNTRY_COLORS[c] for c in df_market['country']]
bars = ax1.barh(df_market['country'], df_market['total_revenue_usd']/1000,
                color=colors_bar, height=0.55, zorder=3)
for bar, val in zip(bars, df_market['total_revenue_usd']):
    ax1.text(val/1000 + 1, bar.get_y()+bar.get_height()/2,
             f'${val/1000:.0f}K', va='center', fontsize=10, fontweight='bold')
ax1.set_xlabel('Total Revenue (USD Thousands)', fontsize=11)
ax1.set_title('Revenue by Market — Full Year 2024', fontsize=11, fontweight='bold', color='#202124')
ax1.grid(axis='x', alpha=0.3)

# Panel 2: Monthly GMV trend
ax2 = fig.add_subplot(gs[0, 1])
ax2.fill_between(df_monthly['month'], df_monthly['gmv_usd']/1000, alpha=0.15, color='#1A73E8')
ax2.plot(df_monthly['month'], df_monthly['gmv_usd']/1000, 'o-',
         color='#1A73E8', linewidth=2.5, markersize=7)
ax2_r = ax2.twinx()
ax2_r.plot(df_monthly['month'], df_monthly['aov'], 's--',
           color='#EA4335', linewidth=1.8, markersize=5, alpha=0.8)
ax2.set_xlabel('Month', fontsize=11)
ax2.set_ylabel('Monthly GMV (USD Thousands)', fontsize=11, color='#1A73E8')
ax2_r.set_ylabel('Average Order Value (USD)', fontsize=11, color='#EA4335')
ax2.set_title('Monthly GMV & AOV Trend', fontsize=11, fontweight='bold', color='#202124')
ax2.set_xticks(range(1,13))
ax2.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'], fontsize=8)
patch1 = mpatches.Patch(color='#1A73E8', label='GMV (USD K)')
patch2 = mpatches.Patch(color='#EA4335', label='AOV (USD)')
ax2.legend(handles=[patch1, patch2], fontsize=9)
ax2.grid(axis='y', alpha=0.3)

# Panel 3: Category revenue + return rate
ax3 = fig.add_subplot(gs[1, 0])
cat_colors = ['#1A73E8','#34A853','#EA4335','#FBBC04','#9C27B0','#00ACC1','#FF7043']
bars3 = ax3.bar(range(len(df_cat)), df_cat['total_revenue_usd']/1000,
                color=cat_colors[:len(df_cat)], zorder=3, alpha=0.85)
ax3_r = ax3.twinx()
ax3_r.plot(range(len(df_cat)), df_cat['return_rate_pct'], 'D--',
           color='#EA4335', linewidth=1.8, markersize=7, label='Return Rate %')
ax3.set_xticks(range(len(df_cat)))
ax3.set_xticklabels([c.replace(' & ',' &\n') for c in df_cat['category']], fontsize=8)
ax3.set_ylabel('Revenue (USD Thousands)', fontsize=11)
ax3_r.set_ylabel('Return Rate (%)', fontsize=11, color='#EA4335')
ax3.set_title('Category Revenue vs Return Rate', fontsize=11, fontweight='bold', color='#202124')
ax3.grid(axis='y', alpha=0.3, zorder=0)
ax3_r.legend(loc='upper right', fontsize=9)

# Panel 4: Operational KPIs — Fulfillment
ax4 = fig.add_subplot(gs[1, 1])
sla_colors = ['#EA4335' if s=='ACTION REQUIRED' else '#34A853' for s in df_ops['sla_status']]
bars4 = ax4.barh(df_ops['country'], df_ops['avg_delivery_days'], color=sla_colors, height=0.5, zorder=3)
ax4.axvline(4, color='#EA4335', linestyle='--', linewidth=1.5, label='SLA Target (4 days)')
for bar, val, breach in zip(bars4, df_ops['avg_delivery_days'], df_ops['sla_breach_pct']):
    ax4.text(val + 0.05, bar.get_y()+bar.get_height()/2,
             f'{val}d  |  {breach}% breach', va='center', fontsize=9, fontweight='bold')
ax4.set_xlabel('Avg Delivery Days', fontsize=11)
ax4.set_title('Fulfillment SLA Performance by Market\n(Red = SLA breach > 4 days)', fontsize=11, fontweight='bold', color='#202124')
ax4.legend(fontsize=9)
ax4.grid(axis='x', alpha=0.3)
ax4.set_xlim(0, 8)

# Panel 5: Channel mix
ax5 = fig.add_subplot(gs[2, 0])
channel_colors = ['#1A73E8','#34A853','#FBBC04']
wedges, texts, autotexts = ax5.pie(
    df_channel['revenue_usd'],
    labels=df_channel['channel'],
    autopct='%1.1f%%',
    colors=channel_colors,
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
for at in autotexts: at.set_fontsize(10); at.set_fontweight('bold')
ax5.set_title(f'Revenue by Channel\nApp AOV: ${df_channel[df_channel.channel=="App"]["aov_usd"].values[0]:.0f} | '
               f'Social: ${df_channel[df_channel.channel=="Social Commerce"]["aov_usd"].values[0]:.0f}',
              fontsize=11, fontweight='bold', color='#202124')

# Panel 6: Social commerce penetration + repeat customer rate
ax6 = fig.add_subplot(gs[2, 1])
x = np.arange(len(df_social))
df_repeat = pd.read_sql_query("""
    SELECT country, ROUND(SUM(is_repeat_customer)*100.0/COUNT(*),1) AS repeat_pct
    FROM orders GROUP BY country ORDER BY repeat_pct DESC
""", conn)
df_combined = df_social.merge(df_repeat, on='country')
bars6a = ax6.bar(x - 0.2, df_combined['social_pct'], 0.35, color='#FBBC04', alpha=0.85, label='Social Commerce %', zorder=3)
bars6b = ax6.bar(x + 0.2, df_combined['repeat_pct'], 0.35, color='#1A73E8', alpha=0.85, label='Repeat Customer %', zorder=3)
ax6.set_xticks(x)
ax6.set_xticklabels(df_combined['country'], fontsize=9)
ax6.set_ylabel('%', fontsize=11)
ax6.set_title('Social Commerce Penetration vs Repeat Customer Rate', fontsize=11, fontweight='bold', color='#202124')
ax6.legend(fontsize=9)
ax6.grid(axis='y', alpha=0.3, zorder=0)

plt.savefig('../outputs/ecomm_ops_dashboard.png', dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
plt.show()
print('Dashboard saved.')

## 9. Executive Summary & GTM Recommendations

In [ ]:
print("="*65)
print("EXECUTIVE SUMMARY — APAC E-commerce Strategy & Operations")
print("="*65)

top_market = df_market.iloc[0]
print(f"""
REVENUE PERFORMANCE
• Total 2024 GMV: ${df['revenue_usd'].sum():,.0f} USD across {df['country'].nunique()} APAC markets
• Top market: {top_market['country']} ({top_market['revenue_share_pct']}% revenue share, AOV ${top_market['aov_usd']})
• Fastest growing: Philippines & Vietnam show strongest Q4 momentum

CATEGORY INSIGHTS
• Electronics drives highest revenue (high AOV) but not the highest volume
• Fashion has highest return rate ({df_cat[df_cat.category=='Fashion']['return_rate_pct'].values[0]}%) — 
  recommend size guide enhancements and virtual try-on features
• Food & Grocery: lowest AOV but high repeat frequency — loyalty programme opportunity

OPERATIONAL PRIORITIES
• Philippines & Indonesia breach 4-day SLA target — logistics partner review needed
• Singapore & Malaysia meeting SLA with room to improve category-level delivery promises

GTM RECOMMENDATIONS FOR H1 2025
1. PHILIPPINES — Invest in logistics infrastructure to close SLA gap;
   Social Commerce revenue growing strongly — increase influencer partnerships
2. VIETNAM — Highest social commerce penetration; scale social-first GTM playbook
3. SINGAPORE — Focus on premium Electronics and BNPL adoption to raise AOV
   (currently underperforming vs market potential)
4. ALL MARKETS — App channel drives highest repeat rates; prioritise 
   app-first features (push notifications, loyalty rewards) to grow LTV
""")

In [ ]:
# Export Tableau-ready flat file
df_tableau = pd.read_sql_query("""
    SELECT *, 
           CASE WHEN returned=1 THEN revenue_usd ELSE 0 END AS return_value_usd
    FROM orders
""", conn)
df_tableau.to_csv('../outputs/ecomm_tableau_ready.csv', index=False)
print(f'Exported {len(df_tableau):,} rows for Tableau.')
conn.close()